In [ ]:
# 在先前貌似过拟合后向网上学习

In [37]:
# 导入相应库
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [39]:
# 数据预处理
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # 随机裁剪成32 * 32， 同时裁剪前添加4像素的填充
    transforms.RandomHorizontalFlip(),         # 随机水平翻转图像， 用于数据增强
    transforms.ToTensor(),                     # 转化为张量
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),   # 官网数据
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])


In [41]:
# 下载训练集及测试集
trainset = torchvision.datasets.CIFAR10(root = './CIFAR10', train = True, download = True, transform = transform_train)
testset = torchvision.datasets.CIFAR10(root = './CIFAR10', train = False, download = True, transform = transform_test)

In [42]:
batch_size = 64   # 先前设128貌似会爆
# 批量读取数据
from torch.utils.data.dataloader import DataLoader
train_loader = torch.utils.data.DataLoader(trainset, batch_size = batch_size, shuffle = True, num_workers = 8, pin_memory = True)
test_loader = torch.utils.data.DataLoader(testset, batch_size = batch_size, shuffle = True, num_workers = 8, pin_memory = True)


In [43]:
# 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torchvision.models.resnet18(weights = None)

# 修改最后一层全连接层以适应CIFAR-10的10个类别
model.fc = nn.Linear(model.fc.in_features, 10)

# 将模型移动到设备上
model = model.to(device)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()         # 交叉熵损失函数
optimizer = optim.Adam(model.parameters(), lr=0.001)   # model.parameters()返回模型所需学习的参数

In [44]:
# 训练函数
def train(epoch):
    model.train()                       # 训练模式
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()    #梯度置零
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)   #获取预测类别，_占位符
        total += labels.size(0)
        correct += (predicted == labels).sum().item()  # 张量转标量

        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {train_loss/(batch_idx+1):.3f}, Acc: {100.*correct/total:.3f}%')

# 测试函数
def test(epoch):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(testloader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Test Epoch: {epoch}, Loss: {test_loss/(batch_idx+1):.3f}, Acc: {100.*correct/total:.3f}%')


In [51]:
# 训练和测试模型
num_epochs = 20
for epoch in range(num_epochs):
    train(epoch)
    test(epoch)

Epoch: 0, Batch: 0, Loss: 0.360, Acc: 88.281%
Epoch: 0, Batch: 100, Loss: 0.433, Acc: 84.715%
Epoch: 0, Batch: 200, Loss: 0.438, Acc: 84.667%
Epoch: 0, Batch: 300, Loss: 0.444, Acc: 84.513%
Test Epoch: 0, Loss: 0.536, Acc: 81.890%
Epoch: 1, Batch: 0, Loss: 0.491, Acc: 83.594%
Epoch: 1, Batch: 100, Loss: 0.418, Acc: 85.481%
Epoch: 1, Batch: 200, Loss: 0.422, Acc: 85.362%
Epoch: 1, Batch: 300, Loss: 0.426, Acc: 85.078%
Test Epoch: 1, Loss: 0.557, Acc: 81.450%
Epoch: 2, Batch: 0, Loss: 0.341, Acc: 87.500%
Epoch: 2, Batch: 100, Loss: 0.392, Acc: 85.791%
Epoch: 2, Batch: 200, Loss: 0.401, Acc: 85.798%
Epoch: 2, Batch: 300, Loss: 0.407, Acc: 85.642%
Test Epoch: 2, Loss: 0.551, Acc: 82.370%
Epoch: 3, Batch: 0, Loss: 0.390, Acc: 82.812%
Epoch: 3, Batch: 100, Loss: 0.422, Acc: 85.025%
Epoch: 3, Batch: 200, Loss: 0.413, Acc: 85.553%
Epoch: 3, Batch: 300, Loss: 0.408, Acc: 85.675%
Test Epoch: 3, Loss: 0.543, Acc: 82.020%
Epoch: 4, Batch: 0, Loss: 0.378, Acc: 85.938%
Epoch: 4, Batch: 100, Loss: 0.

In [ ]:
# 完成后发现确实比我先前的完成的好